# Feature Engineering  

This notebook uses the cleaned data from the folder data/processed and creates additional features.  

In [161]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np 
from ydata_profiling import ProfileReport
from c08_farming_exit import config, features, data_cleaning, mappings, feature_engineering

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [162]:
# #In case you want to run Stata in a cell using the magic command %%stata, initialize it first!
# from c08_farming_exit.stata_utils import init_stata
# init_stata()

## 1. Import processed data

In [163]:
df = pd.read_csv(config.PROCESSED_DATA_DIR / "clean_data.csv")

## 2. Employment features

### 2.1 Employment categories

In [164]:
#EMPLOYMENT CATEGORIES
conditions = [
    (df["farm_empl_last_12_months"] == 1) & (df["empl_type"].isnull()),
    (df["farm_empl_last_12_months"] == 1) & (df["empl_type"].notnull()),
    (df["farm_empl_last_12_months"] == 0) & (df["empl_type"].notnull()),
]
choices = ["only_farm", "hybrid", "fully_off_farm"]

df["empl_category"] = np.select(conditions, choices, default=None)

### 2.2 Work hours per year - absolute numbers

In [165]:
#FARMING
df["cash_crop_hours_per_year"] = np.where(
    df["farm_empl_last_12_months"] == 1,
    (df["farm_empl_cash_crops_duration_rainy_season_in_months_last_12_months"]
     + df["farm_empl_cash_crops_duration_dry_season_in_months_last_12_months"])
    * (df["farm_empl_cash_crops_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["farm_empl_cash_crops_hours_per_day"],
    np.nan
)

df["food_crop_hours_per_year"] = np.where(
    df["farm_empl_last_12_months"] == 1,
    (df["farm_empl_food_crops_duration_rainy_season_in_months_last_12_months"]
     + df["farm_empl_food_crops_duration_dry_season_in_months_last_12_months"])
    * (df["farm_empl_food_crops_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["farm_empl_food_crops_hours_per_day"],
    np.nan
)

# No filtering here: livestock owners that don't do crop farming might not claim that they have worked on the farm. 
df["livestock_hours_per_year"] = (
    (df["farm_empl_livestock_duration_rainy_season_in_months_last_12_months"]
     + df["farm_empl_livestock_duration_dry_season_in_months_last_12_months"])
    * (df["farm_empl_livestock_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["farm_empl_livestock_hours_per_day"]
)

df["farm_empl_hours_per_year"] = df[["cash_crop_hours_per_year", "food_crop_hours_per_year", "livestock_hours_per_year"]].sum(axis=1, min_count=1)

In [166]:
#SELF-EMPLOYMENT
df["self_empl_hours_per_year"] = np.where(
    df["empl_type"] == "Self-employed/own business",
    (df["self_empl_duration_rainy_season_in_months_last_12_months"]
     + df["self_empl_duration_dry_season_in_months_last_12_months"])
    * (df["self_empl_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["self_empl_hours_per_day"],
    np.nan
)

In [167]:
#PERMANENT WAGE EMPLOYMENT
df["wage_empl_permanent_hours_per_year"] = np.where(
    df["wage_empl_type"] == "Permanent",
    12
    * (df["wage_empl_permanent_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["wage_empl_permanent_hours_per_day"],
    np.nan
)

In [168]:
#SEASONAL/CAUSAL WAGE EMPLOYMENT
df["wage_empl_seasonal_casual_hours_per_year"] = np.where(
    df["wage_empl_type"] == "Seasonal",
    (df["wage_empl_seasonal_casual_rainy_season_duration_in_months_last_12_months"]
     + df["wage_empl_seasonal_casual_dry_season_duration_in_months_last_12_months"])
    * (df["wage_empl_seasonal_casual_duration_days_per_week"] * config.WEEKS_PER_MONTH)
    * df["wage_empl_seasonal_casual_duration_hours_per_day"],
    np.nan
)

In [169]:
#TOTAL YEARLY WORK HOURS
cols = [
    "farm_empl_hours_per_year",
    "self_empl_hours_per_year",
    "wage_empl_permanent_hours_per_year",
    "wage_empl_seasonal_casual_hours_per_year",
]
df["total_work_hours_per_year"] = df[cols].sum(axis=1, min_count=1)

### 2.3 Work hours per year - relative numbers

In [170]:
#WORK TYPE SHARES
# avoid dividing by zero -> treat a total of 0 hours as NaN (undefined share)
total_safe = df["total_work_hours_per_year"].replace(0, np.nan)

df["farm_hours_share"] = df["farm_empl_hours_per_year"] / total_safe
df["self_empl_hours_share"] = df["self_empl_hours_per_year"] / total_safe
df["wage_empl_permanent_hours_share"] = df["wage_empl_permanent_hours_per_year"] / total_safe
df["wage_empl_seasonal_casual_hours_share"] = df["wage_empl_seasonal_casual_hours_per_year"] / total_safe


In [171]:
# OFF-FARM WORK SHARE
off_farm_cols = [
    "self_empl_hours_per_year",
    "wage_empl_permanent_hours_per_year",
    "wage_empl_seasonal_casual_hours_per_year",
]

# sum off-farm categories, treating "not applicable" (NaN) as 0 
df["off_farm_hours_per_year"] = df[off_farm_cols].sum(axis=1, min_count=0)

# but if the person has NO work data at all, keep it NaN rather than 0
df.loc[df["total_work_hours_per_year"].isna(), "off_farm_hours_per_year"] = np.nan

# share of total work time that is off-farm
total_safe = df["total_work_hours_per_year"].replace(0, np.nan)
df["off_farm_share"] = df["off_farm_hours_per_year"] / total_safe


### 2.4 Wage per hour - absolute numbers

In [172]:
#FARMING
country_wage_map = {
    "Botswana": feature_engineering.agricultural_wage_per_hour(df, "Botswana",    payment_frequency="Per month",  agriculture_only=False) ,
    "Kenya":    feature_engineering.agricultural_wage_per_hour(df, "Kenya",       payment_frequency="Per day",    agriculture_only=True) ,
    "Namibia":  feature_engineering.agricultural_wage_per_hour(df, "Namibia",     payment_frequency="Per month",  agriculture_only=False) ,
    "Tanzania": feature_engineering.agricultural_wage_per_hour(df, "Tanzania",    payment_frequency="Per day",    agriculture_only=True) ,
    "Zambia":   feature_engineering.agricultural_wage_per_hour(df, "Zambia",      payment_frequency="Per day",    agriculture_only=False) ,
}

condition = df["farm_empl_last_12_months"] == 1

df["farm_empl_wage_per_hour"] = np.where(
    condition,
    df["country"].map(country_wage_map), 
    np.nan
)

df["farm_empl_income_per_year"] = df["farm_empl_wage_per_hour"] * df["farm_empl_hours_per_year"]

In [173]:
#SELF-EMPLOYMENT
input_costs_cols = [
    "self_empl_input_costs_last_30_days",
    "self_empl_labor_costs_last_30_days",
    "self_empl_capital_costs_last_30_days",
]
df["self_empl_input_costs"] = df[input_costs_cols].sum(axis=1, min_count=0)

df["self_empl_wage_per_month"] = df["self_empl_sales_last_30_days"] - df["self_empl_input_costs"]

df["self_empl_hours_per_month"] = np.where(
    df["empl_type"] == "Self-employed/own business",
    (df["self_empl_days_per_week"] * 4)
    * df["self_empl_hours_per_day"],
    np.nan
)

total_safe = df["self_empl_hours_per_month"].replace(0, np.nan)
df["self_empl_wage_per_hour"] = df["self_empl_wage_per_month"] / total_safe

df["self_empl_income_per_year"] = df["self_empl_wage_per_hour"] * df["self_empl_hours_per_year"]

In [174]:
#PERMANENT WAGE EMPLOYMENT
df["wage_empl_permanent_hours_per_month"] = df["wage_empl_permanent_hours_per_year"] / 12

total_safe = df["wage_empl_permanent_hours_per_month"].replace(0, np.nan)
df["wage_empl_permanent_wage_per_hour"] = df["wage_empl_permanent_wage_per_month"] / total_safe

df["wage_empl_permanent_income_per_year"] = df["wage_empl_permanent_wage_per_hour"] * df["wage_empl_permanent_hours_per_year"]

In [ ]:
#SEASONAL/CAUSAL WAGE EMPLOYMENT
df["wage_empl_seasonal_casual_wage_per_hour"] = df.apply(feature_engineering.compute_hourly_wage_for_casual_work, axis=1)

df["wage_empl_seasonal_casual_income_per_year"] = df["wage_empl_seasonal_casual_wage_per_hour"] * df["wage_empl_seasonal_casual_hours_per_year"]

In [ ]:
#TOTAL YEARLY INCOME
cols = [
    "farm_empl_income_per_year",
    "self_empl_income_per_year",
    "wage_empl_permanent_income_per_year",
    "wage_empl_seasonal_casual_income_per_year",
]
df["total_income_per_year"] = df[cols].sum(axis=1, min_count=1)

#### 2.1.3 Employment Drop - TODO: maybe going through the list and checking what I want to keep and not what I want to delete?!

In [ ]:
cols_to_keep = ["total_work_hours_per_year",
                "farm_hours_share"
                "self_empl_hours_share",
                "wage_empl_permanent_hours_share",
                "wage_empl_seasonal_casual_hours_share",
                "off_farm_share",
                
                "farm_empl_wage_per_hour",
                "self_empl_wage_per_hour",
                "wage_empl_permanent_wage_per_hour",
                "wage_empl_seasonal_casual_wage_per_hour"
                "total_income_per_year"
                ]

In [ ]:
# #DROPPING CALCULATION INPUTS
# df = df.drop(columns=[
#     # Farm employment - cash crops
#     "farm_empl_cash_crops_duration_rainy_season_in_months_last_12_months",
#     "farm_empl_cash_crops_duration_dry_season_in_months_last_12_months",
#     "farm_empl_cash_crops_days_per_week",
#     "farm_empl_cash_crops_hours_per_day",
#     "cash_crop_hours_per_year",

#     # Farm employment - food crops
#     "farm_empl_food_crops_duration_rainy_season_in_months_last_12_months",
#     "farm_empl_food_crops_duration_dry_season_in_months_last_12_months",
#     "farm_empl_food_crops_days_per_week",
#     "farm_empl_food_crops_hours_per_day",
#     "food_crop_hours_per_year",
   
#     # Livestock employment
#     "farm_empl_livestock_duration_rainy_season_in_months_last_12_months",
#     "farm_empl_livestock_duration_dry_season_in_months_last_12_months",
#     "farm_empl_livestock_days_per_week",
#     "farm_empl_livestock_hours_per_day",
#     "livestock_hours_per_year",

#     # Farming overall
#     "farm_empl_hours_per_year",
#     "farm_empl_income_per_year",

#     # Self-employment
#     "self_empl_duration_rainy_season_in_months_last_12_months",
#     "self_empl_duration_dry_season_in_months_last_12_months",
#     "self_empl_days_per_week",
#     "self_empl_hours_per_day",
#     "self_empl_input_costs_last_30_days",
#     "self_empl_labor_costs_last_30_days",
#     "self_empl_capital_costs_last_30_days",
#     "self_empl_input_costs",
#     "self_empl_sales_last_30_days",
#     "self_empl_wage_per_month",
#     "self_empl_hours_per_month",
#     "self_empl_hours_per_year",
#     "self_empl_income_per_year",

#     # Wage employment - permanent
#     "wage_empl_permanent_days_per_week",
#     "wage_empl_permanent_hours_per_day",
#     "wage_empl_permanent_hours_per_month",
#     "wage_empl_permanent_wage_per_month",
#     "wage_empl_permanent_hours_per_year",
#     "wage_empl_permanent_income_per_year"

#     # Wage employment - seasonal/casual
#     "wage_empl_seasonal_casual_rainy_season_duration_in_months_last_12_months",
#     "wage_empl_seasonal_casual_dry_season_duration_in_months_last_12_months",
#     "wage_empl_seasonal_casual_duration_days_per_week",
#     "wage_empl_seasonal_casual_duration_hours_per_day",
#     "wage_empl_seasonal_casual_wage_per_interval",
#     "wage_empl_seasonal_casual_payment_frequency",
#     "wage_empl_seasonal_casual_hours_per_year",
#     "wage_empl_seasonal_casual_income_per_year"
#     ])


In [178]:
# df["empl_category"].value_counts(dropna=False)

df['off_farm_share'].notna().sum()



np.int64(3631)

In [179]:
df['self_empl_wage_per_hour'].isna().sum()

np.int64(4606)

In [180]:
# cols=[]
# mask = df["farm_hours_share"].notna() & df["off_farm_share"].notna()
# with pd.option_context("display.max_rows", None, "display.max_columns", None):
#     display(df.loc[mask, cols])
